# 🕵️ El caso de los Retrievers · Solución del reto
**Orquestación de flujos con LangChain y LlamaIndex — Retrievers**
Autora: Elena Díaz - [@HelenDiMo](https://github.com/HelenDiMo)

> ✅ **Este es el notebook con la solución.** Úsalo para el *code review* final, no antes de intentarlo tú.

## Cómo funciona
1. Construyes **4 retrievers** (Parte A).
2. Para cada **pista** (Parte B) eliges qué retriever usar y llamas a `comprobar(...)`.
3. Te dirá ✅ o ❌ **con una pista** si fallas. Puedes intentarlo las veces que quieras.
4. Al resolver las 5, `veredicto()` revela el caso.

⏱ **20 minutos.** Puedes trabajar en parejas. Los desplegables 💡 contienen ayudas: ábrelos solo si te atascas.

## Parte A · Prepara tus retrievers

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Permite importar el paquete `sherlock` (carpeta src/) desde el notebook
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from importlib.metadata import version
from sherlock.embeddings import describir_backend
from sherlock.llm import describir_detective

for paquete in ["langchain-core", "langchain-community", "langchain-classic",
                "llama-index-core", "llama-index-retrievers-bm25"]:
    print(f"{paquete:<30} {version(paquete)}")
print("Embeddings:", describir_backend())
print("Detective :", describir_detective())

from sherlock.vista import mostrar, ids
from sherlock.reto import PISTAS, comprobar, progreso, veredicto

langchain-core                 1.6.9
langchain-community            0.4.2
langchain-classic              1.0.8
llama-index-core               0.14.25
llama-index-retrievers-bm25    0.8.0
Embeddings: spaCy es_core_news_md (vectores de palabras promediados)
Detective : detective simulado (sin LLM)


**Material ya preparado** (es lo mismo que hicimos en el live coding): los documentos y el vector store.

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from sherlock.expediente import cargar_expediente, a_documentos_langchain
from sherlock.embeddings import crear_embeddings_langchain

docs_lc = a_documentos_langchain(cargar_expediente())
vectorstore = InMemoryVectorStore.from_documents(docs_lc, crear_embeddings_langchain())
print(f"{len(docs_lc)} documentos listos")

11 documentos listos


### TODO 1 · Retriever por similitud
Crea `retriever_sim` a partir del `vectorstore`, con **k = 3**.

<details><summary>💡 Ayuda</summary>

Se hace con `vectorstore.as_retriever(search_kwargs={"k": 3})`.
</details>

In [3]:
retriever_sim = vectorstore.as_retriever(search_kwargs={"k": 3})

### TODO 2 · Retriever BM25
Crea `retriever_bm25` con **k = 3**. Usa la función `crear_bm25_langchain` (ya importada abajo), que es el BM25 *con corte* que vimos.

<details><summary>💡 Ayuda</summary>

`crear_bm25_langchain(docs_lc, k=3)`
</details>

In [4]:
from sherlock.retrievers import crear_bm25_langchain

retriever_bm25 = crear_bm25_langchain(docs_lc, k=3)

### TODO 3 · Retriever MMR
Crea `retriever_mmr` con `search_type="mmr"`, **k = 3**, **fetch_k = 6** y **el `lambda_mult` que tú elijas** (entre 0 y 1). Recuerda: `1.0` = solo relevancia, valores más bajos = más variedad.

<details><summary>💡 Ayuda</summary>

Se usa `vectorstore.as_retriever(search_type="mmr", search_kwargs={...})`. Si luego la pista P3 falla, prueba con un `lambda_mult` más bajo.
</details>

In [5]:
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.5},
)

### TODO 4 · Retriever híbrido
Crea `retriever_hibrido` con un `EnsembleRetriever` que combine `retriever_bm25` y `retriever_sim`. Tú eliges los **pesos**.

<details><summary>💡 Ayuda</summary>

⚠️ Se importa de `langchain_classic.retrievers`.

`EnsembleRetriever(retrievers=[...], weights=[..., ...])`. Los pesos deben sumar 1 (por ejemplo `[0.5, 0.5]`).
</details>

In [6]:
from langchain_classic.retrievers import EnsembleRetriever

retriever_hibrido = EnsembleRetriever(
    retrievers=[retriever_bm25, retriever_sim],
    weights=[0.5, 0.5],
)

## Parte B · Las cinco pistas

En cada celda, **sustituye `None`** por la llamada a **uno** de tus cuatro retrievers:
`retriever_sim`, `retriever_bm25`, `retriever_mmr` o `retriever_hibrido`, con `.invoke(PREGUNTA)`.
La pregunta ya está guardada en `PISTAS["P1"]["pregunta"]`, etc.

### P1 · El veneno
Alguien compró la sustancia. Nadie en el expediente la llama "sustancia tóxica"…

<details><summary>💡 Ayuda</summary>

Piensa en qué retriever entiende el **significado** aunque las palabras no coincidan.
</details>

In [7]:
resultado = retriever_sim.invoke(PISTAS["P1"]["pregunta"])
mostrar(resultado[:3], "P1 · " + PISTAS["P1"]["pregunta"])
comprobar("P1", resultado)

P1 · sustancia tóxica
  1. [botica    ] puntuación       - | Botica Pemberton e Hijos. Se despachan diez gramos de extracto de belladona, …
  2. [forense   ] puntuación       - | Lord Edmund Aldridge fue hallado sin vida en la biblioteca de Marlowe Hall. L…
  3. [watson    ] puntuación       - | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…
✅ P1 · El veneno  →  ['botica', 'forense', 'watson']


True

### P2 · El móvil
El apellido del abogado es raro. La carta tiene que salir **la primera**.

<details><summary>💡 Ayuda</summary>

Los nombres propios raros los clava el que busca **palabras exactas**.
</details>

In [8]:
resultado = retriever_bm25.invoke(PISTAS["P2"]["pregunta"])
mostrar(resultado[:3], "P2 · " + PISTAS["P2"]["pregunta"])
comprobar("P2", resultado)

P2 · Whitcombe
  1. [carta     ] puntuación   2.051 | Estimado señor Whitcombe: he descubierto que mi secretario ha falsificado las…
✅ P2 · El móvil  →  ['carta']


True

### P3 · Los testigos repetidos
Hay tres testimonios casi idénticos sobre la luz. No quiero gastar mis 3 resultados en repetir lo mismo.

<details><summary>💡 Ayuda</summary>

Necesitas variedad. Si falla, baja el `lambda_mult` de tu TODO 3.
</details>

In [9]:
resultado = retriever_mmr.invoke(PISTAS["P3"]["pregunta"])
mostrar(resultado[:3], "P3 · " + PISTAS["P3"]["pregunta"])
comprobar("P3", resultado)

P3 · ¿Quién vio luz en la biblioteca?
  1. [cochero   ] puntuación       - | Esperaba en el patio con el coche a las diez y media y vi luz en la ventana d…
  2. [forense   ] puntuación       - | Lord Edmund Aldridge fue hallado sin vida en la biblioteca de Marlowe Hall. L…
  3. [watson    ] puntuación       - | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…
✅ P3 · Los testigos repetidos  →  ['cochero', 'forense', 'watson']


True

### P4 · El perro que no ladró
Una pregunta con dos mitades: un apellido raro ("Whitcombe") **y** un concepto ("el sabueso").

<details><summary>💡 Ayuda</summary>

Una mitad pide palabras exactas y la otra significado. Se necesitan los dos.
</details>

In [10]:
resultado = retriever_hibrido.invoke(PISTAS["P4"]["pregunta"])
mostrar(resultado[:3], "P4 · " + PISTAS["P4"]["pregunta"], puntuaciones=False)
comprobar("P4", resultado)

P4 · Whitcombe y el sabueso
  1. [carta     ] puntuación       - | Estimado señor Whitcombe: he descubierto que mi secretario ha falsificado las…
  2. [jardinero ] puntuación       - | Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el …
  3. [watson    ] puntuación       - | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…
✅ P4 · El perro que no ladró  →  ['carta', 'jardinero', 'watson', 'doncella']


True

### P5 · La coartada rota
Hale dice que pasó la noche en el club, pero algo no cuadra. Hay que traer **a la vez** el registro del club y el telegrama.

<details><summary>💡 Ayuda</summary>

Prueba con varios retrievers y compara. Si el híbrido falla, mira tus pesos.
</details>

In [11]:
resultado = retriever_hibrido.invoke(PISTAS["P5"]["pregunta"])
mostrar(resultado[:3], "P5 · " + PISTAS["P5"]["pregunta"], puntuaciones=False)
comprobar("P5", resultado)

P5 · ¿Dónde pasó la noche Hale?
  1. [club      ] puntuación       - | Libro de salidas, noche del catorce de noviembre. El señor R. Hale firmó su s…
  2. [mayordomo ] puntuación       - | Serví el brandy a las nueve y media, como todas las noches, de la licorera qu…
  3. [telegrama ] puntuación       - | Holmes, el señor Hale asegura que pasó toda la noche en el club Albemarle. Ti…
✅ P5 · La coartada rota  →  ['club', 'mayordomo', 'telegrama', 'doncella']


True

## 🔓 El veredicto
Cuando hayas resuelto las cinco pistas, ejecuta esta celda.

In [12]:
veredicto()

Progreso del caso: 5/5   P1✅ P2✅ P3✅ P4✅ P5✅

╔══════════════════════════════════════════════════════════════════╗
║                         C A S O   R E S U E L T O                ║
╚══════════════════════════════════════════════════════════════════╝

 El culpable es el secretario, R. Hale.

  · MÓVIL     Lord Aldridge había descubierto que falsificaba las cuentas y
              iba a cambiar su testamento (la carta a Whitcombe).
  · MEDIOS    Hale compró diez gramos de belladona cinco días antes (recibo del 9/11).
  · ACCESO    Alguien que el perro conocía y que sabía la rutina de la casa
              envenenó la licorera (jardinero y diario de Watson).
  · COARTADA  Dijo que pasó la noche en el club, pero el registro prueba que
              salió a las 20:45 hacia Marlowe Hall (telegrama vs. club), y el
              mayordomo lo vio salir pálido del despacho tras discutir a gritos.

 Ningún LLM ha sido necesario para resolver el caso: solo traer las pruebas
 correctas. Esa es l

## ⭐ Bonus · Lo mismo en LlamaIndex
¿Te sobra tiempo? Resuelve la pista **P4** con LlamaIndex. Ya tienes el índice y el retriever de similitud preparados.

In [13]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.retrievers import QueryFusionRetriever
from sherlock.expediente import a_documentos_llamaindex
from sherlock.embeddings import crear_embeddings_llamaindex
from sherlock.retrievers import crear_bm25_llamaindex

Settings.embed_model = crear_embeddings_llamaindex()
Settings.llm = None

indice = VectorStoreIndex.from_documents(a_documentos_llamaindex(cargar_expediente()))
nodos = list(indice.docstore.docs.values())

retriever_sim_li = indice.as_retriever(similarity_top_k=3)
retriever_bm25_li = crear_bm25_llamaindex(nodos, k=3)

LLM is explicitly disabled. Using MockLLM.


**TODO Bonus** · Crea `retriever_hibrido_li` con `QueryFusionRetriever` (`num_queries=1`, `mode="reciprocal_rerank"`, `use_async=False`, `similarity_top_k=3`) y compruébalo con P4.

In [14]:
retriever_hibrido_li = QueryFusionRetriever(
    [retriever_sim_li, retriever_bm25_li],
    similarity_top_k=3,
    num_queries=1,
    mode="reciprocal_rerank",
    use_async=False,
)
resultado_li = retriever_hibrido_li.retrieve(PISTAS["P4"]["pregunta"])
mostrar(resultado_li, "LlamaIndex · P4", puntuaciones=False)
comprobar("P4", resultado_li)

LlamaIndex · P4
  1. [jardinero ] puntuación       - | Sombra, el lebrel de la casa, no ladró aquella noche cuando alguien cruzó el …
  2. [carta     ] puntuación       - | Estimado señor Whitcombe: he descubierto que mi secretario ha falsificado las…
  3. [watson    ] puntuación       - | Holmes examinó la licorera con su lupa y concluyó que el veneno se echó allí,…
✅ P4 · El perro que no ladró  →  ['jardinero', 'carta', 'watson']


True

---
# 🔎 Code review y cierre (10 min)

Vamos a revisar el trabajo como si fuera una *pull request*. Empecemos por la pregunta clave: **¿qué retriever acierta qué?**

In [15]:
retrievers = {
    "similitud": retriever_sim,
    "BM25": retriever_bm25,
    "MMR": retriever_mmr,
    "híbrido": retriever_hibrido,
}
from sherlock.reto import evaluar

print(f"{'':<28}" + "".join(f"{nombre:>11}" for nombre in retrievers))
aciertos = {nombre: 0 for nombre in retrievers}
for clave, pista in PISTAS.items():
    fila = ""
    for nombre, r in retrievers.items():
        ok, _ = evaluar(clave, ids(r.invoke(pista["pregunta"])))
        aciertos[nombre] += ok
        fila += f"{'✅' if ok else '❌':>10} "
    print(f"{clave} {pista['titulo']:<24}{fila}")
print(f"{'TOTAL':<28}" + "".join(f"{aciertos[nombre]:>9}/5 " for nombre in retrievers))

                              similitud       BM25        MMR    híbrido
P1 El veneno                        ✅          ❌          ✅          ✅ 
P2 El móvil                         ❌          ✅          ❌          ✅ 
P3 Los testigos repetidos           ❌          ❌          ✅          ❌ 


P4 El perro que no ladró            ❌          ❌          ❌          ✅ 
P5 La coartada rota                 ❌          ✅          ❌          ✅ 
TOTAL                               1/5         2/5         2/5         4/5 


**Lecturas de la tabla** (compruébalas con tu propia ejecución):
* **Ningún retriever simple resuelve las 5 pistas**: cada uno falla donde otro acierta.
* El **híbrido es el que más acierta (4/5)**: arregla el problema de "palabras exactas *o* significado". Pero **falla P3**, que solo resuelve MMR. Moraleja: la *variedad* es un problema distinto, y ningún retriever vale para todo.
* Cuidado al interpretar: estas preguntas están **diseñadas como trampas** para cada método, así que la tabla no mide lo bueno que es cada uno en general. En un proyecto real necesitas muchas más preguntas de prueba, sacadas de usuarios reales, antes de fiarte de un retriever.

## ❌ Errores típicos (qué buscar en un code review)

| # | Error | Cómo se nota | Arreglo |
|---|---|---|---|
| 1 | **`k` mal elegido** | Con `k` bajo falta la prueba clave; con `k` alto entra ruido | Mide con preguntas de prueba, no a ojo |
| 2 | **BM25 "de relleno"** | Devuelve `k` documentos con puntuación 0 y estropea el híbrido | Descartar puntuación 0 (`BM25ConCorte`) |
| 3 | **Culpar al LLM sin mirar el contexto** | Respuestas raras | Imprimir siempre lo que devolvió el retriever |
| 4 | **Palabras que el modelo no conoce** | Similitud baja para todo; resultados "al azar" | `palabras_desconocidas`, añadir BM25 |
| 5 | **Metadatos dentro del embedding (LlamaIndex)** | Títulos y autores cuentan como texto | `excluded_embed_metadata_keys` |
| 6 | **Imports antiguos (LangChain 1.x)** | `ModuleNotFoundError` con `EnsembleRetriever` | `langchain_classic.retrievers` |
| 7 | **Esperar que el híbrido recorte a `k`** | Devuelve más de `k` | Cortar con `[:k]` |
| 8 | **Comparar puntuaciones entre métodos** | BM25, coseno y RRF están en escalas distintas | Comparar posiciones, no valores |

## 🧪 Pregunta de comprobación
Preguntas por **"Whitcombe"** y el retriever de similitud devuelve el telegrama antes que la carta. ¿Por qué pasa, cómo lo has visto en el notebook y qué retriever usarías?

<details><summary>Ver respuesta</summary>

El modelo de embeddings **no conoce la palabra** "Whitcombe" (la lista de `palabras_desconocidas` la delata) y las puntuaciones son bajísimas: el orden es casi ruido. Un retriever **BM25** busca la palabra exacta y pone la carta primera. Si la pregunta además tiene una parte conceptual, un **híbrido**.
</details>

## 📚 Para seguir practicando
* [RAG From Scratch (vídeos cortos de LangChain)](https://www.youtube.com/playlist?list=PLfaIDFEXuae2LXbO1_PKyVJiQ23ZztA0x) · [sus notebooks](https://github.com/langchain-ai/rag-from-scratch)
* [Módulos de retrievers de LlamaIndex](https://developers.llamaindex.ai/python/framework/module_guides/querying/retriever/retrievers/)
* Siguiente nivel: *multi-query*, HyDE y *rerankers*.